In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array

# 1. Read the clean Delta Table you created in Notebook 1
df = spark.table("workspace.default.clean_transactions")

# 2. Smart Data Filter: Only keep CASH_OUT and TRANSFER (where 100% of fraud occurs!)
df_filtered = df.filter(F.col("type").isin("CASH_OUT", "TRANSFER"))

print("Filtered Rows for ML:", df_filtered.count())


Filtered Rows for ML: 2770409


In [0]:
# 1. Split into 80% Training Data and 20% "New/Unseen" Test Data
train_df, test_df = df_filtered.randomSplit([0.8, 0.2], seed=42)

# 2. Convert text column 'type' into a number (0 or 1)
type_indexer = StringIndexer(inputCol="type", outputCol="type_index")

# 3. Combine all numeric clues into a single 'features' vector (required by PySpark MLlib)
feature_cols = [
    "type_index", "amount", "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest", "balance_drop", "is_account_emptied"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# 4. Define the Distributed Random Forest Classifier
rf = RandomForestClassifier(
    labelCol="isFraud",
    featuresCol="features",
    numTrees=30,
    maxDepth=8,
    seed=42
)

# 5. Build & Train the Pipeline on the 80% Training Set
pipeline = Pipeline(stages=[type_indexer, assembler, rf])
model = pipeline.fit(train_df)

print("PySpark Random Forest Model Trained Successfully!")


PySpark Random Forest Model Trained Successfully!


In [0]:
# 1. Run predictions on the 20% Test Data
predictions = model.transform(test_df)

# 2. Measure Model Accuracy (ROC-AUC Score: closer to 1.0 is better)
evaluator = BinaryClassificationEvaluator(labelCol="isFraud", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)
print(f"Model ROC-AUC Accuracy Score: {auc:.4f}")

# 3. Extract the exact Fraud Probability (0.0 to 1.0)
results = predictions.withColumn("fraud_prob", F.round(vector_to_array(F.col("probability"))[1], 4))

# 4. Apply YOUR 90% Threshold Rule!
critical_alerts = results.filter(F.col("fraud_prob") > 0.90) \
    .select("step", "type", "nameOrig", "nameDest", "amount", 
            "oldbalanceOrg", "newbalanceOrig", "balance_drop", 
            "is_account_emptied", "fraud_prob", "isFraud") \
    .orderBy(F.desc("fraud_prob"))

print(f"Total High-Risk Alerts (> 90% Fraud Score): {critical_alerts.count()}")

# 5. Save ALL >90% alerts to a Delta Table for Notebook 3
critical_alerts.write.format("delta").mode("overwrite").saveAsTable("workspace.default.critical_fraud_alerts")

display(critical_alerts.limit(20))


Model ROC-AUC Accuracy Score: 0.9890
Total High-Risk Alerts (> 90% Fraud Score): 560


step,type,nameOrig,nameDest,amount,oldbalanceOrg,newbalanceOrig,balance_drop,is_account_emptied,fraud_prob,isFraud
164,TRANSFER,C138035077,C550830618,1191399.72,1191399.72,0.0,1191399.72,1,0.9766,1
170,TRANSFER,C139567724,C2000039513,871893.15,871893.15,0.0,871893.15,1,0.9766,1
180,TRANSFER,C1508594424,C1444631311,887941.44,887941.44,0.0,887941.44,1,0.9766,1
181,TRANSFER,C997127653,C1969513501,955505.54,955505.54,0.0,955505.54,1,0.9766,1
196,TRANSFER,C983611149,C691637954,1221900.21,1221900.21,0.0,1221900.21,1,0.9766,1
313,TRANSFER,C1279803681,C1617123235,1348095.12,1348095.12,0.0,1348095.12,1,0.9766,1
317,TRANSFER,C257814888,C706563298,1160652.82,1160652.82,0.0,1160652.82,1,0.9766,1
327,TRANSFER,C1987328742,C697444332,1076518.28,1076518.28,0.0,1076518.28,1,0.9766,1
341,TRANSFER,C1428944530,C385537527,929787.24,929787.24,0.0,929787.24,1,0.9766,1
350,TRANSFER,C835019402,C742104229,915672.67,915672.67,0.0,915672.67,1,0.9766,1
